# PrintGuard — YOLOv11s Training

**Model:** YOLOv11s (small — better accuracy than nano, still deployable on RPi 4)  
**Classes:** spaghetti · stringing · warping · layer_shift  
**Dataset:** ~9,300 images merged from 5 public datasets  

## Steps
1. Verify GPU
2. Install dependencies
3. Locate and verify dataset
4. Clean labels (remove invalid/degenerate boxes)
5. Fix `data.yaml` paths for Kaggle
6. Train
7. Evaluate
8. Export to ONNX

## Notes
- ONNX is exported with `half=False` (FP32) for compatibility with ONNX Runtime on RPi ARM
- `save_period=10` saves checkpoints every 10 epochs — surviving disconnects
- Dataset must be attached in Kaggle as: `dejanz/pg-merged-11s`

In [ ]:
# ── CONFIG — edit these if dataset path changes ───────────────────────────────
DATASET_PATH = '/kaggle/input/datasets/dejanz/pg-merged-11s/4c_merged_real_11s'
WORK_DIR     = '/kaggle/working'
MODEL_BASE   = 'yolo11s.pt'
RUN_NAME     = 'printguard_11s'
EPOCHS       = 130
BATCH        = 12
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU found — change runtime type to GPU T4'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# 2. Install dependencies
!pip install ultralytics -q

from ultralytics import YOLO
from pathlib import Path
import yaml, shutil, os
print('Ultralytics ready.')

In [ ]:
# 3. Locate and verify dataset
dataset_root = Path(DATASET_PATH)

yaml_files = list(dataset_root.rglob('data.yaml'))
if not yaml_files:
    raise FileNotFoundError(f'data.yaml not found under {dataset_root}\nCheck DATASET_PATH in the config cell.')

DATA_YAML_SRC = yaml_files[0]
MERGED_ROOT   = DATA_YAML_SRC.parent

print(f'Dataset root : {MERGED_ROOT}')

with open(DATA_YAML_SRC) as f:
    src_config = yaml.safe_load(f)
print(f'Classes ({src_config["nc"]}): {src_config["names"]}')

for split in ['train', 'val', 'test']:
    img_dir = MERGED_ROOT / split / 'images'
    if img_dir.exists():
        n = len(list(img_dir.glob('*.*')))
        print(f'  {split:6s}: {n} images')

In [ ]:
# 4. Clean labels — remove invalid/degenerate bounding boxes
# Kaggle input is read-only, so we copy to /kaggle/working/ first

clean_dir = Path(WORK_DIR) / 'dataset_clean'

if clean_dir.exists():
    shutil.rmtree(clean_dir)
shutil.copytree(MERGED_ROOT, clean_dir)
print(f'Dataset copied to {clean_dir}')

removed = 0
total   = 0

for split in ['train', 'val']:
    labels_dir = clean_dir / split / 'labels'
    if not labels_dir.exists():
        continue

    for txt_file in labels_dir.glob('*.txt'):
        valid_lines = []

        for line in txt_file.read_text().splitlines():
            if not line.strip():
                continue
            total += 1
            parts = line.strip().split()

            try:
                if len(parts) < 5:
                    raise ValueError('too few fields')
                cls_id = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])

                # Valid YOLO bbox: coords normalized [0,1], non-zero size
                if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    raise ValueError('out of range')

                valid_lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')

            except Exception:
                removed += 1
                continue

        txt_file.write_text('\n'.join(valid_lines) + ('\n' if valid_lines else ''))

print(f'Labels scanned : {total}')
print(f'Invalid removed: {removed}')
print('Dataset clean and ready.')

In [ ]:
# 5. Write data.yaml with correct absolute paths for this session

DATA_YAML = Path(WORK_DIR) / 'data.yaml'

config = dict(src_config)   # copy from source
config['path']  = str(clean_dir)
config['train'] = 'train/images'
config['val']   = 'val/images'
config['test']  = 'test/images'

with open(DATA_YAML, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('data.yaml written:')
print(f'  path   : {config["path"]}')
print(f'  classes: {config["nc"]} → {config["names"]}')

In [ ]:
# 6. Train

model = YOLO(MODEL_BASE)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    patience=50,
    device=0,
    workers=4,

    project=str(Path(WORK_DIR) / 'runs'),
    name=RUN_NAME,
    exist_ok=True,
    save_period=10,     # checkpoint every 10 epochs

    # Loss weights — tuned for this dataset
    box=7.5,
    cls=0.22,           # low cls weight — classes are visually distinct
    dfl=1.5,

    # Augmentation
    hsv_h=0.015,        # hue shift handles different filament colors
    hsv_s=0.62,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,         # vertical flip off — camera is always right-side up
    mosaic=0.45,
    copy_paste=0.40,    # copy-paste augmentation for underrepresented classes
    mixup=0.0,
    degrees=3.0,
    translate=0.1,
    scale=0.4,
    shear=1.5,

    # Optimizer
    optimizer='AdamW',
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=5.0,

    val=True,
    plots=True,
)

BEST_PT = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\nTraining complete.')
print(f'Best model: {BEST_PT}')

In [ ]:
# 7. Evaluate best model on validation set

best_model = YOLO(str(BEST_PT))
metrics    = best_model.val(data=str(DATA_YAML), device=0)

print('\n── RESULTS ──────────────────────────────')
print(f'mAP50     : {metrics.box.map50:.4f}')
print(f'mAP50-95  : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')

print('\nPer-class mAP50:')
for name, ap in zip(config['names'], metrics.box.ap50):
    bar = '█' * int(ap * 30)
    print(f'  {name:15s} {ap:.4f}  {bar}')

In [ ]:
# 8. Show training plots

import glob
from IPython.display import Image, display

plot_dir = Path(results.save_dir)
plots    = sorted(plot_dir.glob('*.png'))

if not plots:
    print('No plots found — check save_dir')
else:
    for p in plots:
        print(p.name)
        display(Image(str(p), width=700))

In [ ]:
# 9. Export to ONNX for RPi deployment
#
# Note: half=False (FP32) — ONNX Runtime on RPi ARM does not support FP16.
# The FP32 model is ~18MB which is fine for RPi 4.

onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    half=False,     # FP32 — required for ONNX Runtime on RPi ARM
    simplify=True,
    opset=12,
)

# Copy to /kaggle/working/ root for easy download from Output tab
final_path = Path(WORK_DIR) / 'model.onnx'
shutil.copy(onnx_path, final_path)

size_mb = final_path.stat().st_size / 1e6
print(f'ONNX model : {final_path}')
print(f'Size       : {size_mb:.1f} MB')
print('Download from the Output tab on the right.')